# Mamba v5 - final correlation-first Colab run

This final run restores the stronger v3 architecture at the previously selected 120-candle lookback. It selects an actual trained checkpoint by correlation inside a 1% compounded-MAE quality gate and deploys that exact checkpoint without refitting. The final 30% of validation remains untouched and selects one persistent long/cash entry threshold; positions are carried until the forecast falls to zero, avoiding forced three-candle round trips. Outputs use `mamba_v5_*` names so earlier checkpoints are preserved.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Switch Colab to a GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))

## Fetch the current branch

Push the local v5 changes to `mamba-model` before running this cell. The cell is safe to rerun in an existing Colab runtime.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/ECE1508_GenAI")
branch = "mamba-model"
remote = "https://github.com/WoodyChang21/ECE1508_GenAI.git"

if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", branch], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", branch], check=True)
else:
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", remote, str(repo)], check=True)

os.chdir(repo)
sys.path.insert(0, str(repo))
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## Install model dependencies

`kernels` enables optimized Hugging Face kernels when a compatible build is available. If Colab still prints the sequential-fallback warning, training remains correct but will be slower.

In [ ]:
%pip install -q -r /content/ECE1508_GenAI/requirements-model.txt
%pip install -q kernels "huggingface-hub>=0.34.0,<1.0"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
output_dir = Path("/content/drive/MyDrive/ECE1508_GenAI")
output_dir.mkdir(parents=True, exist_ok=True)
print("Artifacts:", output_dir)

## Verify splits and transformed features

In [ ]:
import pandas as pd
from scripts.models.mamba_data import FEATURE_COLUMNS, FEATURE_SCHEMA_VERSION, load_split

print("Feature schema:", FEATURE_SCHEMA_VERSION)
print("Model features:", len(FEATURE_COLUMNS), FEATURE_COLUMNS)
for name in ["train", "val", "test"]:
    path = repo / "data" / "splits" / f"{name}.parquet"
    frame, values = load_split(path)
    print(name, values.shape, frame["datetime"].min(), frame["datetime"].max(), "finite=", bool(pd.notna(values).all()))

## Optional smoke test

Run this once after changing code. It checks the entire pipeline without replacing the full v5 artifacts.

In [ ]:
# !python scripts/models/train_mamba.py \
#     --device cuda --lookbacks 24 --epochs 1 --d-model 16 --layers 1 \
#     --limit-train 256 --limit-val 96 --limit-test 64 \
#     --output /tmp/mamba_v5_smoke.parquet --checkpoint /tmp/mamba_v5_smoke.pt

## Full v5 training

The single 120-candle model trains for up to 40 epochs. Correlation chooses among checkpoints within 1% of the best compounded-return MAE, and the measured checkpoint is not replaced by a fresh refit. The untouched calibration tail selects only the persistent-policy entry threshold by net compounded return after 1 bp one-way costs; the exit threshold is predeclared at zero.

In [ ]:
!python scripts/models/train_mamba.py \
    --device cuda \
    --lookbacks 120 \
    --forecast-horizon 3 \
    --epochs 40 \
    --patience 10 \
    --batch-size 128 \
    --learning-rate 3e-4 \
    --weight-decay 1e-4 \
    --warmup-epochs 0 \
    --learning-rate-schedule constant \
    --d-model 64 \
    --layers 3 \
    --dropout 0.1 \
    --cumulative-loss-weight 1.0 \
    --direction-loss-weight 0 \
    --calibration-fraction 0.30 \
    --checkpoint-selection-metric correlation \
    --checkpoint-mae-tolerance 0.01 \
    --no-refit-after-selection \
    --strategy-type persistent_long_cash \
    --strategy-exit-threshold-bps 0 \
    --strategy-position-mode long_only \
    --no-strategy-require-all-steps-agree \
    --strategy-thresholds-bps 2 4 6 8 10 \
    --strategy-min-calibration-trades 10 \
    --strategy-selection-metric net_compounded_return \
    --transaction-cost-bps 1 \
    --seed 42 \
    --output "/content/drive/MyDrive/ECE1508_GenAI/mamba_v5_preds.parquet" \
    --checkpoint "/content/drive/MyDrive/ECE1508_GenAI/mamba_v5_checkpoint.pt"

## Final evaluation

Use `locked_strategy` in the resulting JSON as the primary preselected strategy result. Threshold sweeps are retained only as diagnostics. Since the earlier 2024-2025 result informed this redesign, a later unseen period is still needed for a pristine final claim.

In [ ]:
!python scripts/models/evaluate_mamba.py \
    --checkpoint "/content/drive/MyDrive/ECE1508_GenAI/mamba_v5_checkpoint.pt" \
    --predictions-out "/content/drive/MyDrive/ECE1508_GenAI/mamba_v5_eval.parquet" \
    --metrics-out "/content/drive/MyDrive/ECE1508_GenAI/mamba_v5_eval.json" \
    --transaction-cost-bps 1 \
    --thresholds-bps 0 2 4 6 8 10

In [ ]:
import json

report_path = output_dir / "mamba_v5_eval.json"
with report_path.open() as handle:
    report = json.load(handle)

print("Forecast")
display(pd.DataFrame({"Mamba": report["forecast"]["mamba"], **report["forecast"]["baselines"]}).T)
print("Intervals", report["prediction_intervals"])
print("Locked strategy")
display(pd.Series(report.get("locked_strategy", {}).get("test_result", {})))
print("Metadata", report["metadata"])